# FIFA 2026 World Cup Match Predictor

A machine learning model to predict match outcomes for the 2026 FIFA World Cup.
Built using historical international football results from 1872 to present.

## 1. Load Data

In [1]:
import pandas as pd
import os
import kagglehub
import logging

#Ignore KaggleHub logging messages
logging.getLogger('kagglehub').setLevel(logging.ERROR)

#Download datasets from Kaggle
international_football_path = kagglehub.dataset_download('martj42/international-football-results-from-1872-to-2017')
elo_ratings_path = kagglehub.dataset_download('saifalnimri/international-football-elo-ratings')
transfermarket_path = kagglehub.dataset_download('davidcariboo/player-scores')

#From International Football Results dataset
#Load results dataset (contains national team match results from 1872 to 2017)
results_df = pd.read_csv(os.path.join(international_football_path, 'results.csv'))

#From International Football Elo Ratings dataset
#Load Elo ratings dataset (contains national team Elo ratings from 1872 to 2017)
elo_ratings_df = pd.read_csv(os.path.join(elo_ratings_path, 'eloratings.csv'))

#From Football Data from Transfermarkt dataset
#Load national teams dataset (contains national team info like FIFA ranking and market value)
national_teams_df = pd.read_csv(os.path.join(transfermarket_path, 'national_teams.csv'))

print('results_path:', results_df.shape)
print(results_df.head(1))
print("----------------------------------------------------------------------------------------------")
print('elo_ratings_path:', elo_ratings_df.shape)
print(elo_ratings_df.head(1))
print("----------------------------------------------------------------------------------------------")
print('national_teams_path:', national_teams_df.shape)
print(national_teams_df.head(1))

results_path: (49368, 9)
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
----------------------------------------------------------------------------------------------
elo_ratings_path: (6678, 4)
         date     team  rating  change
0  1872-11-30  England  2003.0       3
----------------------------------------------------------------------------------------------
national_teams_path: (118, 17)
   national_team_id        name   team_code  country_id country_name  \
0             10521  San Marino  san-marino         144   San Marino   

  country_code confederation  \
0         SMR1          UEFA   

                                      team_image_url  squad_size  average_age  \
0  https://tmssl.akamaized.net//images/flagge/hea...          24         24.9   

   foreigners_number  foreigners_percentage  total_market_value  c

## 2. Preprocess Data

- Converting dates to datetime format in both datasets
- Fixing non-breaking spaces in the ELO dataset
- Standardizing team names across both datasets so they can be merged
- Filtering matches to 1990 onwards, before today, and dropping friendlies

In [2]:
# Convert dates
results_df['date'] = pd.to_datetime(results_df['date'])
elo_ratings_df['date'] = pd.to_datetime(elo_ratings_df['date'], format='mixed')

# Fix non-breaking spaces in ELO dataset
elo_ratings_df['team'] = elo_ratings_df['team'].str.replace('\xa0', ' ', regex=False)

# Standardize team names across both datasets
name_map = {
    'Cabo Verde': 'Cape Verde',
    'DR Congo': 'Democratic Republic of Congo',
    'Czech Republic': 'Czechia',
    'China PR': 'China',
    'Republic of Ireland': 'Ireland'
}

results_df['home_team'] = results_df['home_team'].replace(name_map)
results_df['away_team'] = results_df['away_team'].replace(name_map)
elo_ratings_df['team'] = elo_ratings_df['team'].replace(name_map)

# Filter results to 1990 onwards, before today, drop friendlies
competitive = results_df[
    (results_df['date'] >= '1990-01-01') &
    (results_df['date'] < pd.Timestamp.today()) &
    (results_df['tournament'] != 'Friendly')
].copy().reset_index(drop=True)

print(competitive.shape)
print(elo_ratings_df.shape)

(21446, 9)
(6678, 4)


## 3. Add Target Variable

Adding the `result` column — the value the model will learn to predict:
- **home_win** — home team scored more goals
- **away_win** — away team scored more goals
- **draw** — both teams scored the same

In [3]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

competitive['result'] = competitive.apply(get_result, axis=1)
print(competitive['result'].value_counts())

result
home_win    10504
away_win     6298
draw         4644
Name: count, dtype: int64


## 4. Feature Engineering

Building features for the model to learn from. 

## 4.1 Last 10 Matches

`get_form_last_10` calculates a team's win rate over their last 10 competitive matches prior to each game. Draws count as 0.5 to reflect the real points system. Returns 0.5 as a neutral default if a team has no prior match history.

In [4]:
def get_form_last_10(team, date, df, n=10):
    # Get all matches for this team before this date
    team_matches = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < date)].sort_values('date').tail(n)    
    
    if len(team_matches) == 0:
        return 0.5  # Neutral form if no match history

    wins = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            if row['result'] == 'home_win':
                wins += 1
            elif row['result'] == 'draw':
                wins += 0.5
        else:  # away team
            if row['result'] == 'away_win':
                wins += 1
            elif row['result'] == 'draw':
                wins += 0.5

    return wins / len(team_matches)  # Return the win percentage

# Apply to every match
competitive['home_form_last_10'] = competitive.apply(
    lambda row: get_form_last_10(row['home_team'], row['date'], competitive), axis=1
)
competitive['away_form_last_10'] = competitive.apply(
    lambda row: get_form_last_10(row['away_team'], row['date'], competitive), axis=1
)

print(competitive[['home_team', 'away_team', 'home_form_last_10', 'away_form_last_10', 'result']])

           home_team    away_team  home_form_last_10  away_form_last_10  \
0           Thailand        Kenya               0.50                0.5   
1           Colombia      Uruguay               0.50                0.5   
2          Indonesia        Kenya               0.50                0.0   
3      United States   Costa Rica               0.50                0.5   
4         Costa Rica      Uruguay               1.00                1.0   
...              ...          ...                ...                ...   
21441        Nigeria     Zimbabwe               0.85                0.3   
21442          India      Jamaica               0.20                0.6   
21443       Zimbabwe        India               0.25                0.2   
21444        Nigeria      Jamaica               0.85                0.6   
21445       Maldives  Afghanistan               0.15                0.3   

         result  
0      home_win  
1      away_win  
2      away_win  
3      away_win  
4      aw

## 4.2 Last 10 Goals Scored and Conceded Averages

`get_goal_avg_last_10` calculates each team's average goals scored and conceded over their last 10 competitive matches prior to each game. This captures both attacking and defensive strength independently of just wins and losses. Returns `(0.0, 0.0)` as a neutral default if a team has no prior match history.

In [5]:
def get_goal_avg_last_10(team, date, df, n=10):
    team_matches = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < date)].sort_values('date').tail(n)
    
    if len(team_matches) == 0:
        return 0.0, 0.0  # Neutral if no match history

    goals_scored = 0
    goals_conceded = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            goals_scored += row['home_score']
            goals_conceded += row['away_score']
        else:  # away team
            goals_scored += row['away_score']
            goals_conceded += row['home_score']
    
    return goals_scored / len(team_matches), goals_conceded / len(team_matches)

#Apply to every match
competitive[['home_goal_avg_last_10', 'home_concede_avg_last_10']] = competitive.apply(
    lambda row: get_goal_avg_last_10(row['home_team'], row['date'], competitive), axis=1, result_type='expand'
)
competitive[['away_goal_avg_last_10', 'away_concede_avg_last_10']] = competitive.apply(
    lambda row: get_goal_avg_last_10(row['away_team'], row['date'], competitive), axis=1, result_type='expand'
)

print(competitive[['home_team', 'away_team', 'home_goal_avg_last_10', 'home_concede_avg_last_10', 'away_goal_avg_last_10', 'away_concede_avg_last_10']])

           home_team    away_team  home_goal_avg_last_10  \
0           Thailand        Kenya                    0.0   
1           Colombia      Uruguay                    0.0   
2          Indonesia        Kenya                    0.0   
3      United States   Costa Rica                    0.0   
4         Costa Rica      Uruguay                    2.0   
...              ...          ...                    ...   
21441        Nigeria     Zimbabwe                    2.3   
21442          India      Jamaica                    0.4   
21443       Zimbabwe        India                    0.8   
21444        Nigeria      Jamaica                    2.1   
21445       Maldives  Afghanistan                    0.7   

       home_concede_avg_last_10  away_goal_avg_last_10  \
0                           0.0                    0.0   
1                           0.0                    0.0   
2                           0.0                    1.0   
3                           0.0                

## 4.3 Last 10 Head to Head Results

`head_to_head_last_10` calculates each team's win rate across their last 10 meetings against each other prior to each game. Unlike recent form which looks at all matches, this captures patterns specific to this matchup — some teams consistently perform better or worse against certain opponents regardless of their overall form. Returns `(0.5, 0.5)` as a neutral default if the two teams have never met before.

In [6]:
def head_to_head_last_10(team1, team2, date, df, n=10):
    #Grabs every match between these two teams before this date, sorted by date, and takes the last 10
    matches = df[
        ((df['home_team'] == team1) & (df['away_team'] == team2)) |
        ((df['home_team'] == team2) & (df['away_team'] == team1))
    ]
    matches = matches[matches['date'] < date].sort_values('date').tail(n)

    # If no matches, return neutral 0.5/0.5
    if len(matches) == 0:
        return 0.5, 0.5

    #Iterate through these matches and count wins for each team
    team1_wins = 0
    team2_wins = 0

    for _, row in matches.iterrows():
        if row['home_team'] == team1:
            if row['result'] == 'home_win':
                team1_wins += 1
            elif row['result'] == 'away_win':
                team2_wins += 1
            else:
                team1_wins += 0.5
                team2_wins += 0.5
        else:  # team2 is home
            if row['result'] == 'home_win':
                team2_wins += 1
            elif row['result'] == 'away_win':
                team1_wins += 1
            else:
                team1_wins += 0.5
                team2_wins += 0.5

    total = len(matches)
    return team1_wins / total, team2_wins / total

competitive[['home_h2h_last_10', 'away_h2h_last_10']] = competitive.apply(
    lambda row: head_to_head_last_10(row['home_team'], row['away_team'], row['date'], competitive),
    axis=1, result_type='expand'
)

print(competitive[['home_team', 'away_team', 'home_h2h_last_10', 'away_h2h_last_10', 'result']].tail(50))

                          home_team                         away_team  \
21396                     Indonesia             Saint Kitts and Nevis   
21397                         Chile                        Cape Verde   
21398                   New Zealand                           Finland   
21399                         Kenya                           Estonia   
21400                        Rwanda                           Grenada   
21401                     Venezuela               Trinidad and Tobago   
21402                    Uzbekistan                             Gabon   
21403                        Guyana                          Dominica   
21404                        Belize                      Sint Maarten   
21405                      Botswana                          Zimbabwe   
21406                       Namibia                           Comoros   
21407                American Samoa                              Guam   
21408                   Puerto Rico      United Sta

## 4.4. ELO Ratings

`get_elo_rating` looks up each team's most recent ELO rating prior to each match from the historical ELO dataset. ELO is a measure of team strength that updates after every match based on the result and the strength of the opponent — a win against a strong team gains more points than a win against a weak one.

Three features are added:
- **home_elo** — ELO rating of the home team going into the match
- **away_elo** — ELO rating of the away team going into the match
- **elo_difference** — difference between the two ratings, positive means the home team is stronger

Teams with no prior ELO history are assigned a default rating of 1500, which sits at the median of the overall distribution.

In [7]:
def get_elo_rating(team, date, df, default=1500):
    #Get all ELO ratings for this team before this data
    team_elo = df[
        (df['team'] == team) &
        (df['date'] < date)
    ].sort_values('date')

    if len(team_elo) == 0:
        return default  # Return default if no ELO history
    
    #Return the most recent ELO rating before this date
    return team_elo.iloc[-1]['rating']

competitive['home_elo'] = competitive.apply(
    lambda row: get_elo_rating(row['home_team'], row['date'], elo_ratings_df), axis=1
)

competitive['away_elo'] = competitive.apply(
    lambda row: get_elo_rating(row['away_team'], row['date'], elo_ratings_df), axis=1
)

#Add ELO difference as a feature (positive means home team is stronger)
competitive['elo_diff'] = competitive['home_elo'] - competitive['away_elo']

print(competitive[['date', 'home_team', 'away_team', 'home_elo', 'away_elo', 'elo_diff']].tail(10))

            date   home_team    away_team  home_elo  away_elo  elo_diff
21436 2026-03-31      Kosovo       Turkey    1714.0    1880.0    -166.0
21437 2026-03-31     Czechia      Denmark    1731.0    1864.0    -133.0
21438 2026-03-31    Cameroon        China    1559.0    1403.0     156.0
21439 2026-03-31   Australia      Curaçao    1774.0    1467.0     307.0
21440 2026-03-31  Kazakhstan      Comoros    1410.0    1359.0      51.0
21441 2026-05-26     Nigeria     Zimbabwe    1627.0    1352.0     275.0
21442 2026-05-27       India      Jamaica    1123.0    1527.0    -404.0
21443 2026-05-30    Zimbabwe        India    1352.0    1123.0     229.0
21444 2026-05-30     Nigeria      Jamaica    1627.0    1527.0     100.0
21445 2026-06-01    Maldives  Afghanistan     841.0    1033.0    -192.0


## 4.5. Tournament Weight

Assigning a weight to each match based on tournament importance. Matches in more prestigious tournaments like the World Cup and continental championships are weighted higher than regional or minor competitions. This is later combined with class weights during training to ensure the model prioritizes learning from the most relevant matches.

In [8]:
tournament_weights = {
    'FIFA World Cup': 3.0,
    'FIFA World Cup qualification': 2.0,
    'Copa América': 2.0,
    'UEFA Euro': 2.0,
    'UEFA Euro qualification': 1.5,
    'African Cup of Nations': 1.5,
    'African Cup of Nations qualification': 1.25,
    'UEFA Nations League': 1.5,
    'Confederations Cup': 1.5,
    'Gold Cup': 1.25,
    'AFC Asian Cup': 1.25,
    'CONCACAF Nations League': 1.25,
}

# Assign weights to each match, default to 1.0 for unlisted tournaments
competitive['match_weight'] = competitive['tournament'].map(tournament_weights).fillna(1.0)

## 4.6 Confederation

Assigning each team their FIFA confederation (UEFA, CONMEBOL, CONCACAF, CAF, AFC, OFC). This gives the model explicit knowledge of regional strength differences — a UEFA team beating a CONCACAF team is less surprising than the reverse.

In [9]:
confederation_map = {
    #UEFA
    'Albania': 'UEFA', 'Andorra': 'UEFA', 'Armenia': 'UEFA', 'Austria': 'UEFA',
    'Azerbaijan': 'UEFA', 'Belarus': 'UEFA', 'Belgium': 'UEFA', 'Bosnia and Herzegovina': 'UEFA',
    'Bulgaria': 'UEFA', 'CIS': 'UEFA', 'Croatia': 'UEFA', 'Cyprus': 'UEFA', 'Czechia': 'UEFA',
    'Czechoslovakia': 'UEFA', 'Czech Republic': 'UEFA', 'Denmark': 'UEFA', 'East Germany': 'UEFA',
    'England': 'UEFA', 'Estonia': 'UEFA', 'Faroe Islands': 'UEFA', 'Finland': 'UEFA',
    'FR Yugoslavia': 'UEFA', 'France': 'UEFA', 'Georgia': 'UEFA', 'German DR': 'UEFA', 'Germany': 'UEFA',
    'Gibraltar': 'UEFA', 'Greece': 'UEFA', 'Hungary': 'UEFA', 'Iceland': 'UEFA', 'Ireland': 'UEFA',
    'Israel': 'UEFA', 'Italy': 'UEFA', 'Kazakhstan': 'UEFA', 'Kosovo': 'UEFA', 'Latvia': 'UEFA',
    'Liechtenstein': 'UEFA', 'Lithuania': 'UEFA', 'Luxembourg': 'UEFA', 'Malta': 'UEFA', 'Moldova': 'UEFA',
    'Montenegro': 'UEFA', 'Netherlands': 'UEFA', 'North Macedonia': 'UEFA', 'Northern Ireland': 'UEFA',
    'Norway': 'UEFA', 'Poland': 'UEFA', 'Portugal': 'UEFA', 'Republic of Ireland': 'UEFA', 'Romania': 'UEFA',
    'Russia': 'UEFA', 'Saarland': 'UEFA', 'San Marino': 'UEFA', 'Scotland': 'UEFA', 'Serbia': 'UEFA', 
    'Serbia and Montenegro': 'UEFA', 'Slovakia': 'UEFA', 'Slovenia': 'UEFA', 'Soviet Union': 'UEFA',
    'Spain': 'UEFA', 'Sweden': 'UEFA', 'Switzerland': 'UEFA', 'Turkey': 'UEFA', 'Ukraine': 'UEFA',
    'Wales': 'UEFA', 'Yugoslavia': 'UEFA',

    #CONMEBOL
    'Argentina': 'CONMEBOL', 'Bolivia': 'CONMEBOL', 'Brazil': 'CONMEBOL', 'Chile': 'CONMEBOL',
    'Colombia': 'CONMEBOL', 'Ecuador': 'CONMEBOL', 'Paraguay': 'CONMEBOL', 'Peru': 'CONMEBOL',
    'Uruguay': 'CONMEBOL', 'Venezuela': 'CONMEBOL',

    #CONCACAF
    'Anguilla': 'CONCACAF', 'Antigua and Barbuda': 'CONCACAF', 'Aruba': 'CONCACAF',
    'Bahamas': 'CONCACAF', 'Barbados': 'CONCACAF', 'Belize': 'CONCACAF',
    'Bermuda': 'CONCACAF', 'Bonaire': 'CONCACAF', 'British Virgin Islands': 'CONCACAF',
    'Canada': 'CONCACAF', 'Cayman Islands': 'CONCACAF', 'Costa Rica': 'CONCACAF',
    'Cuba': 'CONCACAF', 'Curaçao': 'CONCACAF', 'Curacao': 'CONCACAF',
    'Dominica': 'CONCACAF', 'Dominican Republic': 'CONCACAF', 'El Salvador': 'CONCACAF',
    'French Guiana': 'CONCACAF', 'Grenada': 'CONCACAF', 'Guadeloupe': 'CONCACAF',
    'Guatemala': 'CONCACAF', 'Guyana': 'CONCACAF', 'Haiti': 'CONCACAF',
    'Honduras': 'CONCACAF', 'Jamaica': 'CONCACAF', 'Martinique': 'CONCACAF',
    'Mexico': 'CONCACAF', 'Montserrat': 'CONCACAF', 'Nicaragua': 'CONCACAF',
    'Panama': 'CONCACAF', 'Puerto Rico': 'CONCACAF', 'Saint Kitts and Nevis': 'CONCACAF',
    'Saint Lucia': 'CONCACAF', 'Saint Martin': 'CONCACAF',
    'Saint Vincent and the Grenadines': 'CONCACAF', 'Sint Maarten': 'CONCACAF',
    'Suriname': 'CONCACAF', 'Trinidad and Tobago': 'CONCACAF',
    'Turks and Caicos Islands': 'CONCACAF', 'United States': 'CONCACAF',
    'United States Virgin Islands': 'CONCACAF', 'U.S. Virgin Islands': 'CONCACAF',

    #CAF
    'Algeria': 'CAF', 'Angola': 'CAF', 'Benin': 'CAF', 'Botswana': 'CAF',
    'Burkina Faso': 'CAF', 'Burundi': 'CAF', 'Cameroon': 'CAF', 'Cape Verde': 'CAF',
    'Central African Republic': 'CAF', 'Chad': 'CAF', 'Comoros': 'CAF', 'Congo': 'CAF',
    'Democratic Republic of Congo': 'CAF', 'Djibouti': 'CAF', 'DR Congo': 'CAF',
    'Egypt': 'CAF', 'Equatorial Guinea': 'CAF', 'Eritrea': 'CAF', 'Eswatini': 'CAF',
    'Ethiopia': 'CAF', 'Gabon': 'CAF', 'Gambia': 'CAF', 'Ghana': 'CAF',
    'Guinea': 'CAF', 'Guinea-Bissau': 'CAF', 'Ivory Coast': 'CAF', 'Kenya': 'CAF',
    'Lesotho': 'CAF', 'Liberia': 'CAF', 'Libya': 'CAF', 'Madagascar': 'CAF',
    'Malawi': 'CAF', 'Mali': 'CAF', 'Mauritania': 'CAF', 'Mauritius': 'CAF',
    'Morocco': 'CAF', 'Mozambique': 'CAF', 'Namibia': 'CAF', 'Niger': 'CAF',
    'Nigeria': 'CAF', 'Réunion': 'CAF', 'Rwanda': 'CAF', 'São Tomé and Príncipe': 'CAF',
    'Senegal': 'CAF', 'Seychelles': 'CAF', 'Sierra Leone': 'CAF', 'Somalia': 'CAF',
    'South Africa': 'CAF', 'South Sudan': 'CAF', 'Sudan': 'CAF', 'Tanzania': 'CAF',
    'Togo': 'CAF', 'Tunisia': 'CAF', 'Uganda': 'CAF', 'Zambia': 'CAF',
    'Zanzibar': 'CAF', 'Zimbabwe': 'CAF',

    #AFC
    'Afghanistan': 'AFC', 'Australia': 'AFC', 'Bahrain': 'AFC', 'Bangladesh': 'AFC',
    'Bhutan': 'AFC', 'Brunei': 'AFC', 'Cambodia': 'AFC', 'China': 'AFC',
    'Chinese Taipei': 'AFC', 'Guam': 'AFC', 'Hong Kong': 'AFC', 'India': 'AFC',
    'Indonesia': 'AFC', 'Iran': 'AFC', 'Iraq': 'AFC', 'Japan': 'AFC',
    'Jordan': 'AFC', 'Kuwait': 'AFC', 'Kyrgyzstan': 'AFC', 'Laos': 'AFC',
    'Lebanon': 'AFC', 'Macau': 'AFC', 'Malaysia': 'AFC', 'Maldives': 'AFC',
    'Mongolia': 'AFC', 'Myanmar': 'AFC', 'Nepal': 'AFC', 'North Korea': 'AFC',
    'Northern Mariana Islands': 'AFC', 'Oman': 'AFC', 'Pakistan': 'AFC',
    'Palestine': 'AFC', 'Philippines': 'AFC', 'Qatar': 'AFC', 'Saudi Arabia': 'AFC',
    'Singapore': 'AFC', 'South Korea': 'AFC', 'Sri Lanka': 'AFC', 'Syria': 'AFC',
    'Taiwan': 'AFC', 'Tajikistan': 'AFC', 'Thailand': 'AFC', 'Timor-Leste': 'AFC',
    'Turkmenistan': 'AFC', 'United Arab Emirates': 'AFC', 'UAE': 'AFC',
    'Uzbekistan': 'AFC', 'Vietnam': 'AFC', 'Yemen': 'AFC',

    #OFC
    'American Samoa': 'OFC', 'Cook Islands': 'OFC', 'Federated States of Micronesia': 'OFC',
    'Fiji': 'OFC', 'Kiribati': 'OFC', 'New Caledonia': 'OFC', 'New Zealand': 'OFC',
    'Niue': 'OFC', 'Palau': 'OFC', 'Papua New Guinea': 'OFC', 'Samoa': 'OFC', 'Solomon Islands': 'OFC',
    'Tahiti': 'OFC', 'Tonga': 'OFC', 'Tuvalu': 'OFC', 'Vanuatu': 'OFC',
}

competitive['home_confederation'] = competitive['home_team'].map(confederation_map)
competitive['away_confederation'] = competitive['away_team'].map(confederation_map)

# Fill remaining unknowns
competitive['home_confederation'] = competitive['home_confederation'].fillna('Unknown')
competitive['away_confederation'] = competitive['away_confederation'].fillna('Unknown')

print(competitive['home_confederation'].value_counts())

home_confederation
UEFA        5948
CAF         5161
AFC         4683
CONCACAF    3135
CONMEBOL    1249
Unknown      726
OFC          544
Name: count, dtype: int64


## 5 Final Filter

(~~Now that all features have been calculated using the full dataset, we filter down to only matches where both teams are among the 48 qualified World Cup 2026 teams. This gives us a training set that is directly relevant to the tournament while having benefited from the broader match history during feature engineering.~~)

Resetting the dataframe index to start from 0 after filtering and feature engineering.

In [10]:
# wc_teams = [
#     'United States', 'Canada', 'Mexico', 'Argentina', 'Brazil', 'Colombia',
#     'Ecuador', 'Paraguay', 'Uruguay', 'Australia', 'Iran', 'Japan', 'Jordan',
#     'South Korea', 'Qatar', 'Saudi Arabia', 'Uzbekistan', 'Iraq', 'Algeria',
#     'Cabo Verde', 'Ivory Coast', 'Egypt', 'Ghana', 'Morocco', 'Senegal',
#     'South Africa', 'Tunisia', 'DR Congo', 'Curacao', 'Haiti', 'Panama',
#     'New Zealand', 'England', 'France', 'Croatia', 'Norway', 'Portugal',
#     'Germany', 'Netherlands', 'Austria', 'Belgium', 'Scotland', 'Spain',
#     'Switzerland', 'Sweden', 'Turkey', 'Bosnia and Herzegovina', 'Czech Republic'
# ]

# competitive = competitive[
#     competitive['home_team'].isin(wc_teams) &
#     competitive['away_team'].isin(wc_teams)
# ].reset_index(drop=True)

# print(competitive.shape)
# print(competitive['result'].value_counts())
# competitive.tail(10)
competitive = competitive.reset_index(drop=True)

## 6. Encode Target Variable and Categorical Features

Converting text columns to numbers since ML models require numerical inputs. `LabelEncoder` is applied to three columns:
- **result** → `0 = away_win`, `1 = draw`, `2 = home_win`
- **home_confederation** → integer mapping of each confederation
- **away_confederation** → integer mapping of each confederation

In [11]:
from sklearn.preprocessing import LabelEncoder

# Encode Confederation as a number
home_confederation_le = LabelEncoder()
competitive['home_confederation_encoded'] = home_confederation_le.fit_transform(competitive['home_confederation'])
away_confederation_le = LabelEncoder()
competitive['away_confederation_encoded'] = away_confederation_le.fit_transform(competitive['away_confederation'])

# Encode result as a number
result_le = LabelEncoder()
competitive['result_encoded'] = result_le.fit_transform(competitive['result'])

#Show mappings
print(home_confederation_le.classes_)
print(result_le.classes_)
competitive.head()

['AFC' 'CAF' 'CONCACAF' 'CONMEBOL' 'OFC' 'UEFA' 'Unknown']
['away_win' 'draw' 'home_win']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,away_h2h_last_10,home_elo,away_elo,elo_diff,match_weight,home_confederation,away_confederation,home_confederation_encoded,away_confederation_encoded,result_encoded
0,1990-01-29,Thailand,Kenya,2.0,1.0,King's Cup,Bangkok,Thailand,False,home_win,...,0.5,1199.0,1511.0,-312.0,1.0,AFC,CAF,0,1,2
1,1990-02-02,Colombia,Uruguay,0.0,2.0,Miami Cup,Miami,United States,True,away_win,...,0.5,1347.0,1855.0,-508.0,1.0,CONMEBOL,CONMEBOL,3,3,0
2,1990-02-02,Indonesia,Kenya,2.0,3.0,King's Cup,Bangkok,Thailand,True,away_win,...,0.5,1250.0,1511.0,-261.0,1.0,AFC,CAF,0,1,0
3,1990-02-02,United States,Costa Rica,0.0,2.0,Miami Cup,Miami,United States,False,away_win,...,0.5,1586.0,1534.0,52.0,1.0,CONCACAF,CONCACAF,2,2,0
4,1990-02-04,Costa Rica,Uruguay,0.0,2.0,Miami Cup,Miami,United States,True,away_win,...,0.5,1534.0,1855.0,-321.0,1.0,CONCACAF,CONMEBOL,2,3,0


## 7. Train/Test Split

Splitting the data into training (80%) and testing (20%) sets. `shuffle=False` preserves chronological order so the model always trains on older matches and tests on more recent ones — this mirrors how the model will actually be used and prevents data leakage.

In [12]:
from sklearn.model_selection import train_test_split

features = [
    'home_form_last_10',
    'away_form_last_10',
    'home_goal_avg_last_10',
    'home_concede_avg_last_10',
    'away_goal_avg_last_10',
    'away_concede_avg_last_10',
    'home_h2h_last_10',
    'away_h2h_last_10',
    'neutral',
    'home_elo',
    'away_elo',
    'elo_diff',
    'home_confederation_encoded',
    'away_confederation_encoded',
]

X = competitive[features]
y = competitive['result_encoded']

# Time-based split — train on older matches, test on recent ones
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 17156
Testing samples: 4290


## 8. Train the Model — Hyperparameter Tuning

Before training the final model, `RandomizedSearchCV` is used to find the optimal XGBoost hyperparameters. It randomly samples 50 combinations from the parameter grid and evaluates each using 5-fold cross validation, scoring on log loss. Key parameters being tuned:
- **n_estimators** — number of decision trees to build
- **learning_rate** — how much each tree contributes, lower values are more conservative
- **max_depth** — how deep each tree can grow, higher values risk overfitting
- **min_child_weight** — minimum data points required to split a node, helps prevent overfitting
- **subsample** — fraction of training data used per tree, adds randomness to reduce overfitting

In [13]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight

# Compute class weights to balance home_win, draw, away_win distribution
class_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Get tournament weights for training matches only
tournament_weights_train = competitive.loc[X_train.index, 'match_weight'].values

# Combine class weights and tournament weights into a single sample weight
combined_weights = class_weights * tournament_weights_train

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 1.0]
}

xgb = XGBClassifier(
    eval_metric='mlogloss',
    random_state=42
)

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=50,           # tries 50 random combinations
    scoring='neg_log_loss',
    cv=5,                # 5-fold cross validation
    random_state=42,
    n_jobs=-1,           # uses all CPU cores
    verbose=1
)

search.fit(X_train, y_train, sample_weight=combined_weights)

print("Best parameters:", search.best_params_)
print("Best log loss:", -search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.05}
Best log loss: 0.9285760927108042


## 10. Evaluate the Model

Evaluating the final tuned model on the test set using two metrics:
- **Accuracy** — percentage of matches where the model predicted the correct outcome
- **Log Loss** — measures how confident and correct the probability predictions are, lower is better

The classification report breaks down precision, recall and F1 score per outcome. The model achieves **57.18% accuracy** with balanced predictions across all three outcomes — comparable to published football prediction models which typically land in the 55-65% range.

In [ ]:
model = XGBClassifier(
    **search.best_params_,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train, sample_weight=combined_weights)

Accuracy: 57.18%
Log Loss: 0.8903
              precision    recall  f1-score   support

    away_win       0.61      0.60      0.61      1318
        draw       0.29      0.37      0.33       953
    home_win       0.73      0.65      0.69      2019

    accuracy                           0.57      4290
   macro avg       0.54      0.54      0.54      4290
weighted avg       0.60      0.57      0.58      4290



## 9. Evaluate the Model

Evaluating the model on the test set using two metrics:
- **Accuracy** — percentage of matches where the model predicted the correct outcome
- **Log Loss** — measures how confident and correct the probability predictions are, lower is better

The classification report breaks down performance per outcome. The balanced class weights ensure the model learns all three outcomes rather than defaulting to predicting home wins.

In [18]:
from sklearn.metrics import accuracy_score, log_loss, classification_report

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_pred_proba)

print(f"Accuracy: {accuracy:.2%}")
print(f"Log Loss: {loss:.4f}")
print(classification_report(y_test, y_pred, target_names=result_le.classes_))

Accuracy: 57.18%
Log Loss: 0.8903
              precision    recall  f1-score   support

    away_win       0.61      0.60      0.61      1318
        draw       0.29      0.37      0.33       953
    home_win       0.73      0.65      0.69      2019

    accuracy                           0.57      4290
   macro avg       0.54      0.54      0.54      4290
weighted avg       0.60      0.57      0.58      4290

